BLOCK 1: SETUP AND IMPORTS

In [2]:
import requests
import pandas as pd
import json
import time
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


Libraries imported successfully!


BLOCK 2: CONFIGURATION

In [3]:
# European countries
COUNTRIES = {
    "Germany": "DE",
    "France": "FR",
    "Spain": "ES",
    "Italy": "IT",
    "Austria": "AT",
    "Bulgaria": "BG",
    "Belgium": "BE",
    "Greece": "EL",
    "Portugal": "PT",
    "Romania": "RO",
    "Czechia": "CZ",
}

# Technology types
TECH_TYPES = {
    "TOTAL": "Total",
    "HYD": "Hydro",
    "WND_ON": "Wind Onshore",
    "WND_OFF": "Wind Offshore",
    "SUN": "Solar",
    "NUC": "Nuclear",
    "GEO": "Geothermal",
    "BIO": "Biomass",
    "NGAS": "Natural Gas",
    "COAL": "Coal",
    "OIL": "Oil",
}

# Years to analyze
YEARS = [2021, 2022, 2023]

# Price consumer types
PRICE_DATASETS = {
    "nrg_pc_104": "Household Consumers",
    "nrg_pc_204": "Industry Consumers",
    "nrg_pc_304": "All Consumers",
}

print("Configuration set successfully!")
print(f"Countries: {len(COUNTRIES)}")
print(f"Technologies: {len(TECH_TYPES)}")
print(f"Years: {YEARS}")

Configuration set successfully!
Countries: 11
Technologies: 11
Years: [2021, 2022, 2023]


ENTSO E Data 

In [8]:
from entsoe import EntsoePandasClient
import pandas as pd

client = EntsoePandasClient(api_key="7308a46a-2e78-4bf2-a552-3295649e49e3")

eu_countries = ["AT", "BE", "BG", "CZ", "FR", "DE", "GR", "HU", "SI", "ES"]

start = pd.Timestamp("2023-01-02", tz="UTC")
end = pd.Timestamp("2023-01-03", tz="UTC")

rows = []
failed = []

for i, cc in enumerate(eu_countries, start=1):
    print(f"[{i}/{len(eu_countries)}] {cc} ...", end=" ")
    try:
        df = client.query_installed_generation_capacity(cc, start=start, end=end)
    except Exception as e:
        print(f"error: {e}")
        failed.append(cc)
        continue

    if df.empty:
        print("no data")
        failed.append(cc)
        continue

    row = df.iloc[0]
    total_cap = float(row.sum())
    solar_cap = float(row.get("Solar", 0.0))

    rows.append(
        {
            "country": cc,
            "total_capacity_MW": total_cap,
            "solar_capacity_MW": solar_cap,
        }
    )
    print("ok")

print("\nrows length:", len(rows))
print("rows sample:", rows[:3])

country_caps = pd.DataFrame(rows)
print("country_caps columns:", country_caps.columns)
print(country_caps.head())

if not country_caps.empty:
    country_caps["solar_share_pct"] = (
        100 * country_caps["solar_capacity_MW"] / country_caps["total_capacity_MW"]
    )
    print("\nResult:")
    print(
        country_caps[
            ["country", "total_capacity_MW", "solar_capacity_MW", "solar_share_pct"]
        ].sort_values("solar_share_pct", ascending=False)
    )
else:
    print("country_caps is empty – no successful data rows were collected.")
    print("Failed countries:", failed)


[1/10] AT ... ok
[2/10] BE ... ok
[3/10] BG ... ok
[4/10] CZ ... ok
[5/10] FR ... ok
[6/10] DE ... ok
[7/10] GR ... ok
[8/10] HU ... ok
[9/10] SI ... ok
[10/10] ES ... ok

rows length: 10
rows sample: [{'country': 'AT', 'total_capacity_MW': 24625.54, 'solar_capacity_MW': 3265.3}, {'country': 'BE', 'total_capacity_MW': 26799.269999999997, 'solar_capacity_MW': 6474.96}, {'country': 'BG', 'total_capacity_MW': 14343.0, 'solar_capacity_MW': 3092.0}]
country_caps columns: Index(['country', 'total_capacity_MW', 'solar_capacity_MW'], dtype='object')
  country  total_capacity_MW  solar_capacity_MW
0      AT           24625.54            3265.30
1      BE           26799.27            6474.96
2      BG           14343.00            3092.00
3      CZ           19646.00            2083.00
4      FR          143813.63           14638.79

Result:
  country  total_capacity_MW  solar_capacity_MW  solar_share_pct
7      HU           10362.23            3300.37        31.849998
5      DE          233192